# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w05_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same window as ML-04/ML-05 (w03_data_contract.ipynb, w04_feature_leakage_check.ipynb) —
# mid-panel month, never the sealed June 2026 sample.
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
# Same feature vector as ML-05 (w04_feature_leakage_check.ipynb): 5 numeric prev30 features +
# the 4 categoricals its live DESCRIBE confirmed exist on dim_content.
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

CAT_FEATURES = ["content_type", "competition_level", "main_intent", "model_used"]
content_meta = con.sql(f"""
    SELECT content_hash_id, {", ".join(CAT_FEATURES)}
    FROM {TABLES['dim_content']}
""").df()
feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")
for col in CAT_FEATURES:
    feature_frame[col] = feature_frame[col].fillna("unknown")

print(f"Feature frame: {len(feature_frame):,} content items")

# --- Distributions: numeric ---
print(feature_frame[["imp_prev30", "clk_prev30", "avg_position_prev30", "ctr_prev30", "active_days_prev30"]].describe())

# Heavy tails: mean far above median means a few giant pages dominate the raw sum — the classic
# web-traffic shape (skills/auditing-signals/SKILL.md: "distributions first").
for col in ["imp_prev30", "clk_prev30"]:
    print(f"{col}: mean={feature_frame[col].mean():.1f}, median={feature_frame[col].median():.1f}, "
          f"p99={feature_frame[col].quantile(0.99):.1f}, max={feature_frame[col].max():.1f}")

# log1p view of the same two — raw Pearson correlation on either would be dominated by the
# giants and can flip sign after a log transform. Used for any correlation-style test below.
feature_frame["log_imp_prev30"] = np.log1p(feature_frame["imp_prev30"])
feature_frame["log_clk_prev30"] = np.log1p(feature_frame["clk_prev30"])

# --- Distributions: categorical ---
for col in CAT_FEATURES:
    print(f"\n{col} value counts:")
    print(feature_frame[col].value_counts())


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# --- Build the label (identical formula to ML-04/ML-05, w03_data_contract.ipynb /
# w04_feature_leakage_check.ipynb): March impressions falling >20% below prev30. ---
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

overall_rate = data["is_declining"].mean()
print(f"Overall decline rate this slice: {overall_rate:.3f} (n={len(data):,}) — every verdict below is read against this.")

SIZE_FLOOR = 50  # skills/auditing-signals/SKILL.md: no verdict from a bucket under ~50 rows

def bucket_table(df, group_col, label_col="is_declining"):
    t = df.groupby(group_col, observed=True)[label_col].agg(n="count", decline_rate="mean")
    t["insufficient"] = t["n"] < SIZE_FLOOR
    return t

def call_verdict(t, best_bucket, worst_bucket, expected_direction, margin=0.03):
    """expected_direction='up' means decline_rate should be HIGHER in worst_bucket than best_bucket."""
    if t.loc[best_bucket, "insufficient"] or t.loc[worst_bucket, "insufficient"]:
        return "insufficient data (a compared bucket is below the 50-row floor)"
    diff = t.loc[worst_bucket, "decline_rate"] - t.loc[best_bucket, "decline_rate"]
    signed_diff = diff if expected_direction == "up" else -diff
    if signed_diff >= margin:
        return "CONFIRMED"
    if signed_diff <= -margin:
        return "OPPOSITE"
    return "MIXED"

# --- Signal test 1: avg_position_prev30 ---
# Claim: pages ranking worse (deeper avg position) are more likely to be declining.
data["position_bucket"] = pd.cut(
    data["avg_position_prev30"], bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
t1 = bucket_table(data.dropna(subset=["position_bucket"]), "position_bucket")
print("\nSignal test 1 — avg_position_prev30 vs is_declining (claim: worse position -> more decline)")
print(t1)
v1 = call_verdict(t1, best_bucket="top_3", worst_bucket="deep", expected_direction="up")
print(f"Verdict: {v1}")

# --- Signal test 2: ctr_prev30 ---
# Claim: pages with lower CTR (relative to peers this period) are more likely to be declining.
data["ctr_quartile"] = pd.qcut(data["ctr_prev30"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"], duplicates="drop")
t2 = bucket_table(data.dropna(subset=["ctr_quartile"]), "ctr_quartile")
print("\nSignal test 2 — ctr_prev30 vs is_declining (claim: lower CTR -> more decline)")
print(t2)
v2 = call_verdict(t2, best_bucket="Q4_high", worst_bucket="Q1_low", expected_direction="up")
print(f"Verdict: {v2}")

# --- Signal test 3: main_intent ---
# Claim: informational content (broad, high-competition intent) declines more often than
# transactional content (narrower, more durable intent).
t3 = bucket_table(data, "main_intent")
print("\nSignal test 3 — main_intent vs is_declining (claim: informational declines more than transactional)")
print(t3)
if "informational" in t3.index and "transactional" in t3.index:
    v3 = call_verdict(t3, best_bucket="transactional", worst_bucket="informational", expected_direction="up")
else:
    v3 = "insufficient data (informational/transactional category missing this slice)"
print(f"Verdict: {v3}")


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# FlyRank's real product flag `needs_ctr_fix` (a product decision-flag, deliberately NOT part of
# this dataset — see docs/ml-intern-dataset-and-lane-guide.md's product-context section) assumes
# CTR should track ranking position: a page ranking well but clicking poorly signals a
# metadata/title problem worth fixing. That assumption only makes sense if CTR actually falls as
# position gets worse in the first place — this is the test.
ctr_by_position = data.dropna(subset=["position_bucket"]).groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    total_clicks=("clk_prev30", "sum"),
    total_impressions=("imp_prev30", "sum"),
)
# Weighted CTR (total clicks / total impressions), not the mean of per-page CTRs — averaging
# per-row rates would let a handful of tiny pages skew the bucket average.
ctr_by_position["weighted_ctr_pct"] = (ctr_by_position["total_clicks"] / ctr_by_position["total_impressions"] * 100).round(2)
print(ctr_by_position)

POSITION_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
ctr_series = ctr_by_position.reindex(POSITION_ORDER)["weighted_ctr_pct"]
position_ctr_holds = ctr_series.dropna().is_monotonic_decreasing

print(f"\nCTR falls monotonically as position worsens (top_3 -> deep): {position_ctr_holds}")
if position_ctr_holds:
    print("needs_ctr_fix's core assumption (CTR tracks position) holds in this slice — a page with")
    print("unusually low CTR for its position tier is a meaningful signal here, not noise.")
else:
    print("The position -> CTR relationship is not clean in this slice — needs_ctr_fix-style logic")
    print("should be applied with more caution here than the flag's name implies.")


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# Built from whatever the verdicts above actually turned out to be — not a fixed narrative,
# since the direction of each test isn't known until this runs for real.
print("What this means in practice:")

print(f"- Ranking position ({v1}): "
      + ("worse-ranked pages in this slice are meaningfully more likely to be declining — "
         "position is worth weighting in the review queue." if v1 == "CONFIRMED" else
         "position alone did not confirm cleanly — treat it as one input among several, not a "
         "standalone trigger." if v1 == "MIXED" else
         "the opposite of the naive assumption showed up here — worth a second look before "
         "trusting position as a decline signal." if v1 == "OPPOSITE" else
         "not enough data in the compared buckets to call this one either way."))

print(f"- CTR ({v2}): "
      + ("low-CTR pages skew toward decline — a content/metadata review is a reasonable first "
         "move for them." if v2 == "CONFIRMED" else
         "CTR alone did not confirm cleanly — pair it with another signal before acting." if v2 == "MIXED" else
         "the opposite of the naive assumption showed up here — worth a second look." if v2 == "OPPOSITE" else
         "not enough data in the compared buckets to call this one either way."))

print(f"- Content intent ({v3}): "
      + ("informational pages decline more than transactional ones here — intent is worth a "
         "column in the review queue." if v3 == "CONFIRMED" else
         "intent alone did not confirm cleanly on this slice." if v3 == "MIXED" else
         "the opposite of the naive assumption showed up here." if v3 == "OPPOSITE" else
         "not enough data to call this one either way this slice."))

print(f"- needs_ctr_fix's assumption (CTR tracks position): "
      + ("holds in this slice, so a low-CTR page with a good position is a real anomaly worth a "
         "human look, not noise." if position_ctr_holds else
         "does not hold cleanly here, so that flag's logic needs more caution on this data than "
         "its name implies."))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.